# ALIGN — Génération des index FAISS (Flickr8K)

Ce notebook génère les embeddings ALIGN (`kakaobrain/align-base`) pour toutes les images et légendes de Flickr8K, puis les sauvegarde dans des index FAISS prêts à être utilisés par le backend MIR.

**Modèle** : ALIGN ViT-B/32 — EfficientNet-B7 (images) + BERT-Large (texte)  
**Dataset** : Flickr8K — 8 091 images, 40 455 légendes  
**Index produits** : `index_images_align.faiss` + `index_captions_align.faiss`  
**Dimension** : 640


In [ ]:
!pip install -q transformers faiss-cpu Pillow tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import faiss
import torch
from PIL import Image
from tqdm import tqdm
from transformers import AlignProcessor, AlignModel
import matplotlib.pyplot as plt

# ── Chemins (adapter si besoin) ────────────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/Projet_MIR/aPartie 2"
IMAGES_PATH  = os.path.join(DRIVE_ROOT, "dataset/archive (Unzipped Files)/Images")
CAPTIONS_FILE = os.path.join(DRIVE_ROOT, "dataset/archive (Unzipped Files)/captions.txt")
SAVE_DIR     = os.path.join(DRIVE_ROOT, "save")   # même dossier que les index CLIP

os.makedirs(SAVE_DIR, exist_ok=True)

# ── GPU si disponible ──────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Chargement du modèle ALIGN

In [ ]:
print("Chargement de kakaobrain/align-base ...")
processor = AlignProcessor.from_pretrained("kakaobrain/align-base")
model     = AlignModel.from_pretrained("kakaobrain/align-base").to(device).eval()
print("Modèle prêt.")

# Dimension de l'embedding
DIM = model.config.projection_dim
print(f"Dimension des embeddings : {DIM}")

## 2. Chargement des métadonnées Flickr8K

In [ ]:
df          = pd.read_csv(CAPTIONS_FILE)
unique_imgs = df["image"].unique()   # même ordre que le notebook CLIP

print(f"Images uniques : {len(unique_imgs)}")
print(f"Légendes totales : {len(df)}")
df.head(3)

## 3. Encodage des images → index FAISS

Chaque image est encodée avec `model.get_image_features()`, normalisée L2, puis ajoutée à un `IndexFlatIP` (produit scalaire = cosinus sur vecteurs normalisés).

In [ ]:
BATCH_IMG = 32   # réduire à 16 si OOM

def encode_images_batch(img_names, batch_size=BATCH_IMG):
    all_vecs = []
    for i in tqdm(range(0, len(img_names), batch_size), desc="Images"):
        batch_names = img_names[i : i + batch_size]
        imgs = []
        for name in batch_names:
            path = os.path.join(IMAGES_PATH, name)
            try:
                imgs.append(Image.open(path).convert("RGB"))
            except Exception:
                imgs.append(Image.new("RGB", (224, 224)))  # image noire si manquante
        inputs = processor(images=imgs, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            vecs = model.get_image_features(**{k: v for k, v in inputs.items() if k in ["pixel_values"]})
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        all_vecs.append(vecs.cpu().float().numpy())
    return np.vstack(all_vecs)

img_vectors = encode_images_batch(unique_imgs)
print(f"Shape image vectors : {img_vectors.shape}")

In [ ]:
# Création et remplissage de l'index images
index_images = faiss.IndexFlatIP(img_vectors.shape[1])
index_images.add(img_vectors)
print(f"Index images : {index_images.ntotal} vecteurs")

save_path_imgs = os.path.join(SAVE_DIR, "index_images_align.faiss")
faiss.write_index(index_images, save_path_imgs)
print(f"Sauvegardé → {save_path_imgs}")

## 4. Encodage des légendes → index FAISS

In [ ]:
BATCH_TXT = 256

def encode_texts_batch(texts, batch_size=BATCH_TXT):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Captions"):
        batch = texts[i : i + batch_size]
        inputs = processor(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64,
        ).to(device)
        with torch.no_grad():
            vecs = model.get_text_features(**{k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask", "token_type_ids"]})
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        all_vecs.append(vecs.cpu().float().numpy())
    return np.vstack(all_vecs)

captions     = df["caption"].tolist()
cap_vectors  = encode_texts_batch(captions)
print(f"Shape caption vectors : {cap_vectors.shape}")

In [ ]:
index_captions = faiss.IndexFlatIP(cap_vectors.shape[1])
index_captions.add(cap_vectors)
print(f"Index captions : {index_captions.ntotal} vecteurs")

save_path_caps = os.path.join(SAVE_DIR, "index_captions_align.faiss")
faiss.write_index(index_captions, save_path_caps)
print(f"Sauvegardé → {save_path_caps}")

## 5. Vérification rapide — Text → Image

In [ ]:
def query_text_to_image(text, k=5):
    inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
    with torch.no_grad():
        vec = model.get_text_features(**{k_: v for k_, v in inputs.items() if k_ in ["input_ids", "attention_mask", "token_type_ids"]})
        vec = vec / vec.norm(dim=-1, keepdim=True)
    scores, indices = index_images.search(vec.cpu().float().numpy(), k)
    return [
        {"filename": unique_imgs[idx], "score": float(s)}
        for idx, s in zip(indices[0], scores[0])
    ]

results = query_text_to_image("A dog running on the beach", k=4)
fig, axes = plt.subplots(1, len(results), figsize=(16, 4))
for ax, r in zip(axes, results):
    ax.imshow(Image.open(os.path.join(IMAGES_PATH, r["filename"])))
    ax.set_title(f"Score: {r['score']:.3f}", fontsize=9)
    ax.axis("off")
plt.suptitle('Query: "A dog running on the beach"', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Vérification rapide — Image → Text

In [ ]:
def query_image_to_text(image_idx, k=5):
    vec = index_images.reconstruct(image_idx).reshape(1, -1)
    scores, indices = index_captions.search(vec, k)
    return [
        {"caption": df.iloc[int(idx)]["caption"], "score": float(s)}
        for idx, s in zip(indices[0], scores[0])
    ]

test_idx = 42
img_name = unique_imgs[test_idx]
print(f"Image : {img_name}")
plt.figure(figsize=(4, 3))
plt.imshow(Image.open(os.path.join(IMAGES_PATH, img_name)))
plt.axis("off")
plt.show()

for i, r in enumerate(query_image_to_text(test_idx, k=5), 1):
    print(f"{i}. [{r['score']:.3f}] {r['caption']}")

## 7. Récapitulatif des fichiers générés

In [ ]:
for fname in ["index_images_align.faiss", "index_captions_align.faiss"]:
    path = os.path.join(SAVE_DIR, fname)
    size_mb = os.path.getsize(path) / 1e6
    print(f"{fname:40s}  {size_mb:.1f} MB")

print()
print("Copier ces deux fichiers dans MIR_Project/indexes_faiss/")